# v5 article extraction — scored against the v5 gold article list

The frozen v5 pipeline (tag `extraction-v5`) on the **held-out v5 test volumes** (Penny Magazine 1844, Saturday
Magazine 1837, Chambers's Edinburgh Journal 1840 — three issues each, never used to write rules) scored against
`gold_article_list_test_v5.csv`, which Owen annotated from the printed pages alone, without seeing the output
(guide: `documentation/article_extraction_pipeline_iterative_build/gold_article_list_annotation_guide.md`).
The gold list was corrected afterwards from the printed page only: Saturday gold 50 is issue 291 (entered as 281;
the masthead reads "Nº 291"). The output read is `output/article_extraction/v5/`, frozen. Page numbers are PDF
pages; the check images are in `output/article_extraction/v5/<volume>/reading_order/`. What v5 changed is in
`documentation/article_extraction_pipeline_iterative_build/v5_decisions.md`; the issue-start fixes made after this
test are in `v6_decisions.md` (v6 has not been tested on new held-out texts).

**Notes on this test set.**
- The Chambers volume is an American reprint of the journal ("Two Dollars per Annum"), numbered in Roman
  numerals (CV–CVII) on the dates of the Edinburgh Nos. 414–416; flagged for later.
- Most issue numbers and many dates here are wrong for three masthead forms (section 8 and `v5_decisions.md`,
  "Held-out test"); article boundaries are not affected by them.

**How items are matched.** Each gold item is located in the output by its first and last words
(OCR-tolerant matching on the item's own pages). When the same opening words also appear earlier, quoted inside
another article, an equally good match at an article's start wins. Gold issue numbers may be given in Roman
numerals; they are compared by value. Every number below is computed by `score_gold.py`; nothing is judged by hand.

The v5 gold list has one `notes` column (figure "fig:" entries and item notes together); section 10 sets the
notes beside each error.

| Outcome | Meaning |
|---|---|
| **exact** | the item opens a pipeline article of its own, and its last words are in that article |
| **merged** | its pipeline article also holds the start of the gold item before or after it (a missed boundary) |
| **split** | its first and last words land in different pipeline articles (a false boundary inside it) |
| **lost** | its first words are in no article's text (set aside as furniture, absorbed into other matter, or not found) |


In [1]:
import sys
from pathlib import Path

import pandas as pd

pd.set_option("display.width", 180)
pd.set_option("display.max_colwidth", 70)
pd.set_option("display.max_rows", 200)

HERE = next(p for p in [Path("."), Path("evaluation/evaluating_v5_article_extraction")]
            if (p / "gold_article_list_test_v5.csv").exists())
SCORER_DIR = HERE.resolve().parent / "evaluating_v2_article_extraction"  # score_gold.py lives there
sys.path.insert(0, str(SCORER_DIR))
from score_gold import load_gold, load_pipeline, score, summary_tables, write, VOLUME_ALIASES

VERSION = "v5"
GOLD = HERE / "gold_article_list_test_v5.csv"
OUT_DIR = HERE  # results: <VERSION>_gold_comp_eval_*.csv / _summary.md

## Run the scoring and save the results

In [2]:
gold = load_gold(GOLD)
res = score(VERSION, gold)
summary_path = write(res, VERSION, OUT_DIR, GOLD)
items, arts, pieces, imprints = res["items"], res["pipeline_articles"], res["pieces"], res["imprints"]
tables = summary_tables(res)
own = items[items.opens_own_article]
print(f"Gold list: {len(gold)} rows — {gold.article_type.value_counts().to_dict()}")
print(f"Scored {len(items)} gold items against {len(arts)} pipeline articles ({VERSION})")
print(f"Wrote {OUT_DIR}/{VERSION}_gold_comp_eval_*.csv and {summary_path.name}")

Gold list: 95 rows — {'article': 87, 'imprint': 6, 'editoral_notice': 2}
Scored 87 gold items against 96 pipeline articles (v5)
Wrote ./v5_gold_comp_eval_*.csv and v5_gold_comp_eval_summary.md


## 1. Headline: items and boundaries

In [3]:
tables['items']

,chambers_1840,saturday_1837,penny_1844,all
gold items,38,29,20,87
pipeline articles,46,32,18,96
exact,31/38 (82%),21/29 (72%),16/20 (80%),68/87 (78%)
merged,2/38 (5%),4/29 (14%),4/20 (20%),10/87 (11%)
split,3/38 (8%),4/29 (14%),0/20 (0%),7/87 (8%)
lost,2/38 (5%),0/29 (0%),0/20 (0%),2/87 (2%)
boundary recall,35/38 (92%),27/29 (93%),18/20 (90%),80/87 (92%)
boundary precision,35/46 (76%),27/32 (84%),18/18 (100%),80/96 (83%)
extra pipeline articles,11,5,0,16


In [4]:
# Item outcome counts per volume
pd.crosstab(items.volume, items.status, margins=True, margins_name="total")

status,exact,lost,merged,split,total
volume,,,,,
chambers_1840,31,2,2,3,38
penny_1844,16,0,4,0,20
saturday_1837,21,0,4,4,29
total,68,2,10,7,87


## 2. Successes — items extracted exactly

In [5]:
exact = items[items.status == "exact"]
print(f"{len(exact)} of {len(items)} gold items extracted exactly")
exact[["volume", "gold_id", "gold_title", "pipeline_title", "start_article_id"]]

68 of 87 gold items extracted exactly


,volume,gold_id,gold_title,pipeline_title,start_article_id
0,chambers_1840,1,The Crock of Gold.,THE CROCK OF GOLD,chambers_1840_0002
1,chambers_1840,2,Supposed Changes in the Weather of Late Years.,to SUPPOSED CHANGES IN THE WEATHER OF LATE YEARS,chambers_1840_0003
2,chambers_1840,3,The Silk-Wool of the Alpaca.,THE SILK-WOOL OF THE ALPACA,chambers_1840_0004
3,chambers_1840,4,Picture of a German Student.,PICTURE OF A GERMAN STUDENT,chambers_1840_0005
4,chambers_1840,5,Isaac Watts.,ISAAC WATTS,chambers_1840_0006
5,chambers_1840,6,Accidental Discovery of the Steam Engine,ACCIDENTAL DISCOVERY OF THE STEAM-ENGINE,chambers_1840_0010
6,chambers_1840,7,The Scotch Costumes of London Artists.,THE SCOTCH COSTUMES OF LONDON ARTISTS,chambers_1840_0011
7,chambers_1840,8,What Constitutes a Patent.,WHAT CONSTITUTES A PATENT,chambers_1840_0012
8,chambers_1840,9,Welees and Dervishes.,WELEES AND DERVISHES,chambers_1840_0013
11,chambers_1840,12,Mariana.,MARIANA,chambers_1840_0015


## 3. Errors — merged items (missed boundaries)

In [6]:
items[items.status.str.startswith("merged")][
    ["volume", "gold_id", "start_pdf_page", "gold_title", "merged_with", "start_article_id", "pipeline_title"]]

,volume,gold_id,start_pdf_page,gold_title,merged_with,start_article_id,pipeline_title
9,chambers_1840,10,14,Yankee Pedlar.,11,chambers_1840_0014,YANKEE PEDLAR
10,chambers_1840,11,14,The Daguerreotype.,10,chambers_1840_0014,
45,saturday_1837,52,13,The Town of Bern.,53,saturday_1837_0010,IN SWITZERLAND
46,saturday_1837,53,16,"The Indian Rubber Tree, (Usquahuil.) II.",52,saturday_1837_0010,
49,saturday_1837,56,17,"I. On The Ancient Inhabitants of Europe, and Their Subsequent Divi...",57,saturday_1837_0014,
50,saturday_1837,57,18,,56,saturday_1837_0014,
72,penny_1844,81,14,Hudibras.—No. I.,82,penny_1844_0006,HUDIBRAS.-No. I
73,penny_1844,82,16,Practical Philosophy in a Small Way.,81,penny_1844_0006,
80,penny_1844,89,25,Trees of Port Philip.,90,penny_1844_0013,Trees of Port Phillip
81,penny_1844,90,25,Population of the United States to the Square Mile.,89,penny_1844_0013,


## 4. Errors — split items (false boundaries inside an item)

In [7]:
split = items[items.status.str.contains("split")]
split[["volume", "gold_id", "start_pdf_page", "gold_title", "start_article_id", "end_article_id", "pipeline_articles"]]

,volume,gold_id,start_pdf_page,gold_title,start_article_id,end_article_id,pipeline_articles
14,chambers_1840,17,18,Labour Songs,chambers_1840_0019,chambers_1840_0020,chambers_1840_0019;chambers_1840_0020
23,chambers_1840,28,24,Literary Formulas,chambers_1840_0030,chambers_1840_0031,chambers_1840_0030;chambers_1840_0031
26,chambers_1840,31,26,Specimens of the English Sonnet Writers.,chambers_1840_0033,chambers_1840_0034,chambers_1840_0033;chambers_1840_0034
38,saturday_1837,44,5,"Gilbert White, of Selborne.",saturday_1837_0001,saturday_1837_0002,saturday_1837_0001;saturday_1837_0002
40,saturday_1837,46,7,Spiders. I.,saturday_1837_0004,saturday_1837_0005,saturday_1837_0004;saturday_1837_0005
54,saturday_1837,61,19,Notes on Forest Trees. No. VI.,saturday_1837_0018,saturday_1837_0019,saturday_1837_0018;saturday_1837_0019
56,saturday_1837,64,22,On Winter.,saturday_1837_0020,saturday_1837_0021,saturday_1837_0020;saturday_1837_0021


## 5. Errors — lost items (opening words not in any article's text)

In [8]:
items[items.status == "lost"][["volume", "gold_id", "start_pdf_page", "gold_title", "start_found_in", "start_score"]]

,volume,gold_id,start_pdf_page,gold_title,start_found_in,start_score
25,chambers_1840,30,26,Reason for Mental Cultivation.,not found,0.0
28,chambers_1840,33,27,Stage-Coach Journey to the Tournament.,back_matter,1.0


## 6. Extra pipeline articles (hold no gold item's start)

Usually fragments of split items (`gold_ends` shows a gold item whose *last* words fall in it) or page
furniture taken as an article (masthead lines, prices, dates).

In [9]:
extra = arts[arts.status == "extra"]
print(extra.volume.value_counts().to_dict())
extra[["volume", "article_id", "pipeline_title", "pages", "word_count", "gold_ends", "text_start"]]

{'chambers_1840': 11, 'saturday_1837': 5}


,volume,article_id,pipeline_title,pages,word_count,gold_ends,text_start
0,chambers_1840,chambers_1840_0001,STORIES OF THE IRISH PEASANTRY,8,12,,""" Mind not high things: but condescend to men of low estate."""
6,chambers_1840,chambers_1840_0007,HAPPY FRAILTY,12,603,,"""How meanly dwells the immortal bodies are! How vile these Why was"
7,chambers_1840,chambers_1840_0008,THE ROSE,12,158,,How fair is the rose! what a beautiful flower. The glory of
8,chambers_1840,chambers_1840_0009,THE ANT OR EMMET,12,825,,Those Emmets-how little they are in our eyes! and a troop of
15,chambers_1840,chambers_1840_0016,NOTICE,15,253,,THE Publisher of the American Edition of this popular periodical a...
19,chambers_1840,chambers_1840_0020,NaN,19,1695,17,"All prosp'rous seems beneath the earth, Full and kindly fill'd the..."
28,chambers_1840,chambers_1840_0029,NOTICE,23;24,271,,THE Publisher of the American Edition of this popular periodical a...
30,chambers_1840,chambers_1840_0031,"SATURDAY, JANUARY 18, 1840",24;25,2094,28,"into any one workhouse, they would find the inmates in the enjoyment"
33,chambers_1840,chambers_1840_0034,DRUMMOND TO HIS LUTE,27,116,31,"My lute, be as thou wert when thou did grow With thy"
35,chambers_1840,chambers_1840_0036,STAGE-COACH JOURNEY TO THE TOURNAMENT,27;28,1167,33,"At length we found ourselves in motion, and we began to beguile"


## 7. Fields — summary

In [10]:
tables['fields']

,chambers_1840,saturday_1837,penny_1844,all
items scored (open their own article),35,27,18,80
title,33/35 (94%),21/27 (78%),17/18 (94%),71/80 (89%)
front matter pieces in place,3/7 (43%),0/4 (0%),1/1 (100%),4/12 (33%)
back matter pieces in place,11/14 (79%),10/17 (59%),5/9 (56%),26/40 (65%)
series (where either has one),2/6 (33%),1/5 (20%),2/2 (100%),5/13 (38%)
continued from,33/35 (94%),25/27 (93%),18/18 (100%),76/80 (95%)
to be continued,35/35 (100%),22/27 (81%),18/18 (100%),75/80 (94%)
editorial notice / article,35/35 (100%),27/27 (100%),18/18 (100%),80/80 (100%)
illustration count,34/35 (97%),24/27 (89%),17/18 (94%),75/80 (94%)
issue number,0/35 (0%),0/27 (0%),14/18 (78%),14/80 (18%)


### 7a. Titles that disagree

In [11]:
own[~own.title_result.isin(["correct", "both untitled"])][
    ["volume", "gold_id", "start_pdf_page", "gold_title", "pipeline_title", "title_result"]]

,volume,gold_id,start_pdf_page,gold_title,pipeline_title,title_result
15,chambers_1840,18,19,Snatches of Continental Recollections.,HOW THEY CURE NE'ERDOWEELS,wrong
17,chambers_1840,20,20,Greek Supersitions,SKETCHES OF SUPERSTITIONS,wrong
38,saturday_1837,44,5,"Gilbert White, of Selborne.",VIEW OF SELBORNE,wrong
45,saturday_1837,52,13,The Town of Bern.,IN SWITZERLAND,wrong
49,saturday_1837,56,17,"I. On The Ancient Inhabitants of Europe, and Their Subsequent Divi...",,missing
55,saturday_1837,63,21,The Alligator.,THE LARCH,wrong
62,saturday_1837,70,25,A Charade.,TMHARADE,wrong
64,saturday_1837,72,26,II. Mythology of the Nursery.,,missing
70,penny_1844,79,13,"Description of Paris in 1579, by a Venetian Ambassador.",,missing


### 7b. Front and back matter pieces

Each gold piece is found **in place** (the pipeline's front/back matter), **in text** (left in the article text),
elsewhere, or **missing** (set aside or not found). Footnotes — many in Chambers — are counted separately; some
give sources, others explain a word or concept.

In [12]:
tables['pieces']

result,in place,in text,in title,missing,total
kind,,,,,
back matter,20,9,0,1,30
back matter (footnote),6,2,0,2,10
front matter,4,3,1,4,12
total,30,14,1,7,52


In [13]:
# Pieces not in place
pieces[pieces.result != "in place"][["volume", "gold_id", "field", "footnote", "result", "piece"]]

,volume,gold_id,field,footnote,result,piece
0,chambers_1840,1,front_matter,False,missing,By Mrs S. C. Hall. “mind not high things: but condescend to men of...
3,chambers_1840,10,back_matter,False,missing,Captain Marryat’s Diary in America.
4,chambers_1840,12,front_matter,False,in text,“Mariana in the moated grange.”—Measure for Measure.
11,chambers_1840,18,front_matter,False,in title,How They Cure Ne’erdoweels.
13,chambers_1840,24,back_matter,False,in text,"Notice of Mainzer’s Sketches of Music, in the Athenæum."
16,chambers_1840,31,front_matter,False,in text,"Scorn not the sonnet; critic, you have frowned, Mindless of its ju..."
17,chambers_1840,31,back_matter,True,missing,"* Match, or lover. * Warbling, from Ramage, French."
21,saturday_1837,44,back_matter,False,in text,P.S.— The writer of the foregoing article had not access to the or...
22,saturday_1837,45,back_matter,False,in text,Chappelow.
23,saturday_1837,46,back_matter,False,in text,[Kirby’s Bridgewater Treatise.]


In [14]:
# Pipeline front/back matter with no gold counterpart (extra pieces), per item
x = own[(own.front_matter_extra > 0) | (own.back_matter_extra > 0)]
x[["volume", "gold_id", "gold_title", "front_matter_extra", "front_matter_pipeline",
   "back_matter_extra", "back_matter_pipeline"]]

,volume,gold_id,gold_title,front_matter_extra,front_matter_pipeline,back_matter_extra,back_matter_pipeline
13,chambers_1840,16,“Adventures of Robin Day.”,0.0,,1.0,"• Please, sir, saving conscious of the enormity of happy I, beginn..."
14,chambers_1840,17,Labour Songs,0.0,SECOND ARTICLE.,2.0,"* Sweet, 'mid the tresses of the bride, Blooms the virgin coronal,..."
21,chambers_1840,24,Taste for Music in Germany.,0.0,,1.0,"** On our road to the dwelling which had been pointed out to us, w..."
24,chambers_1840,29,Walks out of Town.,0.0,BY HUGH MILLAR. | AUTHOR OF THE TRADITIONARY HISTORY OF CROMARTY. ...,1.0,"Dugald Sure , Can Stewart"
26,chambers_1840,31,Specimens of the English Sonnet Writers.,0.0,,1.0,"* * * We left our home, with a merry friend, one fine morning, a f..."
49,saturday_1837,56,"I. On The Ancient Inhabitants of Europe, and Their Subsequent Divi...",0.0,,1.0,Domesticated Animals
54,saturday_1837,61,Notes on Forest Trees. No. VI.,0.0,,1.0,"MEN should consider, that the more they enjoy, they are accountabl..."
55,saturday_1837,63,The Alligator.,0.0,,1.0,"MEN should consider, that the more they enjoy, they are accountabl..."
63,saturday_1837,71,,0.0,,2.0,"The Young Lady's Friend | THE POPULAR SUPERSTITIONS, LEGENDS, AND ..."
72,penny_1844,81,Hudibras.—No. I.,0.0,,2.0,"* ""His puissant sword unto his side, tied; Near his undaunted hear..."


### 7c. Series and continuations that disagree

In [15]:
g = gold.set_index(["volume", "gold_id"])
s = own[~own.series_result.isin(["correct", "none (both)"])].copy()
s["gold_series"] = [g.loc[(v, i), "series"] for v, i in zip(s.volume, s.gold_id)]
s[["volume", "gold_id", "gold_title", "gold_series", "pipeline_series", "series_result"]]

,volume,gold_id,gold_title,gold_series,pipeline_series,series_result
0,chambers_1840,1,The Crock of Gold.,Stories of the Irish Peasantry.,,missing
15,chambers_1840,18,Snatches of Continental Recollections.,,Snatches of Continental Recollections,extra
17,chambers_1840,20,Greek Supersitions,Sketches of Superstitions.,,missing
23,chambers_1840,28,Literary Formulas,,Cvii,extra
45,saturday_1837,52,The Town of Bern.,,The Town of Bern,extra
49,saturday_1837,56,"I. On The Ancient Inhabitants of Europe, and Their Subsequent Divi...","The Popular Superstitions, Legends, and Fiction, of the Middle Ages.",,missing
54,saturday_1837,61,Notes on Forest Trees. No. VI.,,Notes on Forest Trees,extra
64,saturday_1837,72,II. Mythology of the Nursery.,"The Popular Superstitions, Legends, and Fiction, of the Middle Ages.",,missing


In [16]:
own[~own.continued_from_result.str.startswith("correct") | ~own.to_be_continued_result.str.startswith("correct")][
    ["volume", "gold_id", "gold_title", "continued_from_result", "to_be_continued_result"]]

,volume,gold_id,gold_title,continued_from_result,to_be_continued_result
24,chambers_1840,29,Walks out of Town.,missing,correct (no)
29,chambers_1840,34,"Greek Superstitions Continued — Oracles, Sacrifices, Omens.",missing,correct (no)
40,saturday_1837,46,Spiders. I.,correct (no),missing
41,saturday_1837,47,Notes on Forest Trees. No. V.,correct (yes),missing
43,saturday_1837,49,Australian Grammar. III.,correct (yes),missing
53,saturday_1837,60,Spiders. II.,missing,missing
54,saturday_1837,61,Notes on Forest Trees. No. VI.,missing,missing


### 7d. Illustrations and article type that disagree

In [17]:
own[own.figures_result != "correct"][["volume", "gold_id", "gold_title", "figures_gold", "figures_pipeline"]]

,volume,gold_id,gold_title,figures_gold,figures_pipeline
23,chambers_1840,28,Literary Formulas,0.0,1.0
44,saturday_1837,50,Crowns and Coronets.,1.0,2.0
54,saturday_1837,61,Notes on Forest Trees. No. VI.,2.0,3.0
55,saturday_1837,63,The Alligator.,1.0,2.0
74,penny_1844,83,Roman Lighthouse at Boulogne.,0.0,1.0


In [18]:
own[~own.type_result.str.startswith("correct")][["volume", "gold_id", "gold_title", "article_type", "type_result"]]

,volume,gold_id,gold_title,article_type,type_result


## 8. Issue metadata — number and date

In [19]:
issue = own.groupby(["volume", "issue_no"]).agg(
    pipeline_issue_no=("pipeline_issue_no", lambda x: ", ".join(sorted(set(map(str, x))))),
    pipeline_issue_date=("pipeline_issue_date", lambda x: ", ".join(sorted(set(map(str, x))))),
    number_correct=("issue_no_result", lambda x: (x == "correct").mean().round(2)),
    date_correct=("issue_date_result", lambda x: (x == "correct").mean().round(2)))
issue

pipeline_issue_no pipeline_issue_date  number_correct  date_correct
volume        issue_no                                                                    
chambers_1840 105                                 01-04-1840             0.0           1.0
              106                                 01-11-1840             0.0           1.0
              107                                 01-11-1840             0.0           0.0
penny_1844    755                 4, 755        , 01-06-1844             0.2           0.8
              756                    756          01-13-1844             1.0           1.0
              757                    757          01-20-1844             1.0           1.0
saturday_1837 291                                 01-21-1837             0.0           0.0
              292                                 01-21-1837             0.0           1.0
              293                                 01-21-1837             0.0           0.0

## 9. Imprints (publishing history)

In [20]:
if "imprints" in tables:
    display(tables["imprints"])
imprints[["volume", "gold_id", "issue_no", "result", "missing_part_found_in", "pipeline_imprint"]]

result,complete,missing,partial,total
volume,,,,
chambers_1840,0,2,1,3
saturday_1837,1,2,0,3
total,1,4,1,6


,volume,gold_id,issue_no,result,missing_part_found_in,pipeline_imprint
0,chambers_1840,14,105,missing,not found,
1,chambers_1840,27,106,partial,box role: furniture,"In Weekly Numbers, price $200; or, in Monthly Parts, price $2 25 p..."
2,chambers_1840,43,107,missing,text,
3,saturday_1837,51,291,complete,,"LONDON: Published by JOHN WILLIAM PARKER, WEST STRAND; and sold by..."
4,saturday_1837,62,292,missing,box role: figure,
5,saturday_1837,75,293,missing,box role: imprint,


## 10. Errors beside Owen's notes

Every item that is not exact, or that opens its own article with a field wrong, with its note from the gold list.


In [21]:
notes = gold.set_index(["volume", "gold_id"])["notes"]
fields = ["title_result", "series_result", "figures_result"]
wrong = items[(items.status != "exact") | (items.opens_own_article & (
    ~items.title_result.isin(["correct", "both untitled"]) | ~items.series_result.isin(["correct", "none (both)"])
    | (items.figures_result == "wrong")))].copy()
wrong["notes"] = [notes.get((v, i), "") for v, i in zip(wrong.volume, wrong.gold_id)]
with pd.option_context("display.max_colwidth", 200):
    display(wrong[["volume", "gold_id", "start_pdf_page", "status", "gold_title", "pipeline_title", *fields, "notes"]])


,volume,gold_id,start_pdf_page,status,gold_title,pipeline_title,title_result,series_result,figures_result,notes
0,chambers_1840,1,8,exact,The Crock of Gold.,THE CROCK OF GOLD,correct,missing,correct,
9,chambers_1840,10,14,merged,Yankee Pedlar.,YANKEE PEDLAR,correct,none (both),correct,
10,chambers_1840,11,14,merged,The Daguerreotype.,,NaN,NaN,NaN,
14,chambers_1840,17,18,split,Labour Songs,LABOUR SONGS,correct,none (both),correct,verse included in this article
15,chambers_1840,18,19,exact,Snatches of Continental Recollections.,HOW THEY CURE NE'ERDOWEELS,wrong,extra,correct,
17,chambers_1840,20,20,exact,Greek Supersitions,SKETCHES OF SUPERSTITIONS,wrong,missing,correct,
23,chambers_1840,28,24,split,Literary Formulas,LITERARY FORMULAS,correct,extra,wrong,
25,chambers_1840,30,26,lost,Reason for Mental Cultivation.,,NaN,NaN,NaN,
26,chambers_1840,31,26,split,Specimens of the English Sonnet Writers.,SPECIMENS OF THE ENGLISH SONNET WRITERS,correct,none (both),correct,"lots of verse in article, example of footnote at end of column with two article corresponding to top article not closest."
28,chambers_1840,33,27,lost,Stage-Coach Journey to the Tournament.,,NaN,NaN,NaN,


## 11. Look up one gold item

In [22]:
def show(volume, gold_id):
    """A gold row next to what the pipeline produced for it."""
    row = gold[(gold.volume == volume) & (gold.gold_id == gold_id)].iloc[0]
    it = items[(items.volume == volume) & (items.gold_id == gold_id)]
    print("GOLD:", {k: v for k, v in row.items() if v and k not in ("pipeline_volume", "notes")})
    if len(it):
        it = it.iloc[0]
        print("\nRESULT:", it.status, "| title:", it.get("title_result", ""), "| merged with:", it.merged_with or "-")
        pipe = load_pipeline(VERSION, VOLUME_ALIASES.get(volume, volume))["articles"]
        ids = str(it.get("pipeline_articles") or it.start_article_id or "").split(";")
        for aid in [a for a in ids if a in pipe.index]:
            a = pipe.loc[aid]
            print(f"\n{aid}: {a.title!r}  pages {a.pages}")
            print("  front:", a.front_matter if isinstance(a.front_matter, str) else "-")
            print("  text :", " ".join(str(a.text).split()[:30]), "…")
            print("  back :", a.back_matter if isinstance(a.back_matter, str) else "-")

show("saturday_1837", 52)  # The Town of Bern: merged with the Indian Rubber Tree

GOLD: {'gold_id': 52, 'volume': 'saturday_1837', 'issue_no': '292', 'issue_date': '1.21.1837', 'start_pdf_page': 13, 'start_printed_page': '18', 'start_column': 'left', 'end_pdf_page': '16', 'article_type': 'article', 'title': 'The Town of Bern.', 'first_words': 'Bern, the capital of', 'last_words': 'miles from the city.', 'front_matter': 'In Switzerland.', 'back_matter': '* See Saturday Magazine, Vol. V., p. 234.', 'end_page': 16}

RESULT: merged | title: wrong | merged with: 53

saturday_1837_0010: 'IN SWITZERLAND'  pages 14;15;16
  front: -
  text : BERN, the capital of the canton of the same name, is one of the largest and most populous towns in Switzerland. Its origin is altogether modern: it was founded in …
  back : -
